In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import gc
import os
import sys
import gymnasium as gym
import torch

torch.set_num_threads(1)
gc.collect()

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import MatchEnv, PoolController, RandomController
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import RandomReset
from src.rl.reward_shapers import DenseReward_3
from src.rl.trainer import train_ppo, export_huggingface

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

⚡ Device: cuda


In [ ]:
# ── CONFIGURATION ──
STAGE = 3
SAVE_DIR = f"models/stage{STAGE}"
POOL_DIR = f"models/stage{STAGE}/pool"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(POOL_DIR, exist_ok=True)

MAX_STEPS = 1800  # 30.0s
TIME_LIMIT = 30.0
NUM_ENVS = 16


def make_env(env_idx: int):
    def _init():
        import torch
        torch.set_num_threads(1) 
        
        is_red = env_idx % 2 == 0
        learner_team = "red" if is_red else "blue"
        opp_team = "blue" if is_red else "red"

        opp_ctrl = PoolController(pool_dir=POOL_DIR, device="cpu")

        if is_red:
            roster = [
                PlayerSlot("red", PlayerStats(name="Learner", accel=3200.0), controller="RL"),
                PlayerSlot("blue", PlayerStats(name="Opponent", accel=3200.0), controller=opp_ctrl),
            ]
        else:
            roster = [
                PlayerSlot("red", PlayerStats(name="Opponent", accel=3200.0), controller=opp_ctrl),
                PlayerSlot("blue", PlayerStats(name="Learner", accel=3200.0), controller="RL"),
            ]

        cfg = MatchConfig(mode=ClassicMatchMode(time_limit=TIME_LIMIT, score_limit=99), roster=roster)

        return MatchEnv(
            match_config=cfg,
            reward_shaper=DenseReward_3(team=learner_team),
            reset_strategy=RandomReset(),
            learner_team=learner_team,
            max_steps=MAX_STEPS,
        )
    return _init

# Use AsyncVectorEnv with spawn to engage all CPU cores safely
train_envs = gym.vector.AsyncVectorEnv(
    [make_env(i) for i in range(NUM_ENVS)],
    context="spawn" 
)

model = ActorCritic(obs_dim=80).to(device)

stage2_best = "models/stage2/best_model.pt"
if os.path.exists(stage2_best):
    model.load_state_dict(torch.load(stage2_best, map_location=device, weights_only=False))
    print(f"✅ Loaded checkpoint from Stage 2")

✅ Loaded checkpoint from Stage 2


In [ ]:
train_ppo(
    envs=train_envs,
    model=model,
    device=device,
    max_steps=MAX_STEPS,
    time_limit=TIME_LIMIT,
    baseline_type="heuristic",
    double_eval=True,              
    previous_model_path=stage2_best, 
    total_timesteps=30_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir=SAVE_DIR,
    pool_dir=POOL_DIR,
    lr_initial=3e-5,
    lr_final=5e-6,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef_initial=0.006,
    ent_coef_final=0.002,
)

train_envs.close()

In [15]:
if os.path.exists(f"{SAVE_DIR}/best_model.pt"):
    export_huggingface(
        model_path=f"{SAVE_DIR}/best_model.pt",
        save_dir=SAVE_DIR,
        device=device,
        stage=STAGE,
        time_limit=TIME_LIMIT,
        max_steps=MAX_STEPS,
        eval_episodes=100
    )

🧪 Running rigorous 100-episode validation against HEURISTIC...
✅ Hugging Face assets safely exported to models/stage3/


In [17]:
if os.path.exists(f"models/stage1/best_model.pt"):
    export_huggingface(
        model_path=f"models/stage1/best_model.pt",
        save_dir="models/stage1",
        device=device,
        stage=1,
        time_limit=TIME_LIMIT,
        max_steps=MAX_STEPS,
        eval_episodes=100
    )

🧪 Running rigorous 100-episode validation against RANDOM...
✅ Hugging Face assets safely exported to models/stage1/


In [18]:
if os.path.exists(f"models/stage2/best_model.pt"):
    export_huggingface(
        model_path=f"models/stage2/best_model.pt",
        save_dir="models/stage2",
        device=device,
        stage=2,
        time_limit=TIME_LIMIT,
        max_steps=MAX_STEPS,
        eval_episodes=100
    )

🧪 Running rigorous 100-episode validation against HEURISTIC...
✅ Hugging Face assets safely exported to models/stage2/


In [21]:
import os
import sys
import torch

sys.path.insert(0, os.path.abspath(".."))
from src.rl.evaluator import evaluate_and_generate_html

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Render 5 evaluation matches of your best checkpoint
replay_file = evaluate_and_generate_html(
    model_or_path="models/stage3/best_model.pt",
    device=device,
    baseline_type="heuristic",  # Visualizes matches against Heuristic bot
    output_dir="render/",
    filename="stage3_diagnostic.html",
    num_episodes=10,
    max_steps=1800,  # 30 seconds per match
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/training/render/stage3_diagnostic.html
